# **PyTorch Tutorial for the model structure**
This tutorial notebook aims at getting you familiar with PyTorch models. It contains:

> 1. How to use components in **torch.nn** to define layers learned in Lesson 07 & 08.
> 2. How to combine your layers into a complete model (torch.nn.Module).
> 3. How does a data sample go through your model.
> 4. What is torch.nn.Sequential? How to define your model with torch.nn.Sequential?
> 5. What is a loss function? How can it be implemented in PyTorch?

In [1]:
"""
Step 1: Import crucial packages
"""
import torch
from torch import nn

In [2]:
"""
Step 2: Define several layers based on torch.nn
Recall the principle of convolution and MLP. Explain the reason why the input's shape is changing.
"""
x = torch.randn(16, 1, 28, 28)

n_filter = 16
filter_size = 2
cnn_layer1 = nn.Conv2d(in_channels=1, out_channels=n_filter, kernel_size=filter_size, padding='same')

y1 = cnn_layer1(x)

n_filter = 32
filter_size = 4
cnn_layer2 = nn.Conv2d(in_channels=1, out_channels=n_filter, kernel_size=filter_size, padding='same')

y2 = cnn_layer2(x)

for ii in [x, y1, y2]:
    print(ii.size())


pooling_layer = nn.MaxPool2d(kernel_size=2)
y2 = pooling_layer(y1)

for ii in [x, y1, y2]:
    print(ii.size())

torch.Size([16, 1, 28, 28])
torch.Size([16, 16, 28, 28])
torch.Size([16, 32, 28, 28])
torch.Size([16, 1, 28, 28])
torch.Size([16, 16, 28, 28])
torch.Size([16, 16, 14, 14])


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/conv.py:549: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1036.)
  return F.conv2d(


In [3]:
"""
Step 3: Use torch.nn.Module to define a complete model
Generally, we use __init__ to define layers in our model; forward(self, x) to define how the model will work
"""
class MNISTModel(nn.Module):
    def __init__(self, in_shape, out_shape):
        super(MNISTModel, self).__init__()
        self.fc1 = nn.Linear(in_shape, 512)
        self.fc2 = nn.Linear(512, 512)
        self.fc3 = nn.Linear(512, 256)
        self.fc4 = nn.Linear(768, out_shape)

    def forward(self, x):
        # try to draw the working flow of this model
        x = torch.relu(self.fc1(x))
        x1 = torch.relu(self.fc2(x))
        x2 = torch.relu(self.fc3(x))
        x3 = torch.cat((x1, x2), dim=1)
        x = torch.softmax(self.fc4(x3), dim=1)
        return x

model = MNISTModel(784, 10)
print(model)

input_data = torch.randn(64, 784)
output = model(input_data)
print(output.size())

MNISTModel(
  (fc1): Linear(in_features=784, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=512, bias=True)
  (fc3): Linear(in_features=512, out_features=256, bias=True)
  (fc4): Linear(in_features=768, out_features=10, bias=True)
)
torch.Size([64, 10])


In [5]:
"""
Step 4: Use torch.nn.Sequential to define a complete model
Generally, we use __init__ to define layers in our model; forward(self, x) to define how the model will work
"""

model = nn.Sequential(
    nn.Linear(in_features=784, out_features=128),
    nn.ReLU(),
    nn.Linear(in_features=128, out_features=10),
    nn.Softmax(dim=1)
)

print(model)

input_data = torch.randn(64, 784)

output = model(input_data)

print(output.size())


Sequential(
  (0): Linear(in_features=784, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=10, bias=True)
  (3): Softmax(dim=1)
)
torch.Size([64, 10])


# Question
What is the difference between torch.nn.Module and torch.nn.Sequential?

In [6]:
"""
Step 5: Define loss functions in PyTorch
"""
# mse loss
input = torch.randn(5, 2)
target = torch.randn(5,2)

mse_loss = nn.MSELoss()
loss = mse_loss(input, target)

print(loss.item())


# cross entropy loss
predictions = torch.randn(5, 10)
target = torch.randint(10, (5,))
cross_entropy_loss = nn.CrossEntropyLoss()
loss = cross_entropy_loss(predictions, target)
print(loss.item())


# bce loss
predictions = torch.randn(5, requires_grad=True)
print(predictions)
probabilities = torch.sigmoid(predictions)
print(probabilities)
target = torch.randint(0, 2, (5,)).float()
print(target)
bce_loss = nn.BCELoss()
loss = bce_loss(probabilities, target)

print(loss.item())

1.5224833488464355
2.601069450378418
tensor([ 0.4032, -1.0806, -0.2269, -0.4663, -0.9035], requires_grad=True)
tensor([0.5995, 0.2534, 0.4435, 0.3855, 0.2883], grad_fn=<SigmoidBackward0>)
tensor([1., 0., 1., 1., 0.])
0.5820677280426025
